# Lab 2: Training & Transfer

How a model trains often matters more than how large it is. Three small routines
carry most of that weight: the optimizer update that moves each parameter, the
normalization that keeps activations well scaled, and the surgery that reuses a
pretrained backbone on a new task. You write all three, then put them to work in
a controlled transfer study on CIFAR-10.

**What you will do**

1. Apply one momentum-SGD update and watch the velocity build across steps.
2. Turn a pretrained backbone into a transfer model by replacing its head and freezing the rest.
3. Standardize activations with a batch-normalization forward pass.
4. Design a classifier head to a parameter budget, then run one fine-tuning change.

The red **Your Task** banners mark the parts you complete. Everything else is
provided and ready to run.


## At a glance

| | |
|---|---|
| Stack | PyTorch and torchvision; a GPU runtime is recommended |
| You write | `sgd_step`, `build_transfer_model`, `batchnorm_forward` |
| You submit | Five values, R1 through R5, from the final cell |
| GPU runtime | A few minutes of training in quick mode; longer on the full split |


<div style="border-left:6px solid #A31F34;background:#fff5f6;color:#111827;padding:14px 18px;border-radius:8px;margin:18px 0;">
<div style="color:#A31F34;font-weight:700;text-transform:uppercase;letter-spacing:0.06em;font-size:0.78rem;">Big picture</div>
<p style="margin:6px 0 0;">A training pipeline is a chain of small, checkable steps. When each step is correct on a fixed probe, the full run is far more likely to behave the way you expect.</p>
</div>


## What is graded

R1 through R3 check your three functions on fixed inputs. R4 accepts your head
design as long as its parameter count lands in the stated range. R5 is a
pass-or-fail flag on your experimental setup. You never submit a measured
accuracy; you read it to interpret your runs. The graded numbers are the same in
quick and full mode, and the official CIFAR-10 test split stays untouched.


<div style="border-left:6px solid #B45309;background:#fffbeb;color:#111827;padding:14px 18px;border-radius:8px;margin:18px 0;">
<div style="color:#B45309;font-weight:700;text-transform:uppercase;letter-spacing:0.06em;font-size:0.78rem;">Watch out</div>
<p style="margin:6px 0 0;">Do not enter a validation accuracy from your run as an answer. Accuracy drifts with hardware and library versions, so only the deterministic checks R1 through R5 are graded.</p>
</div>


In [ ]:
from __future__ import annotations

import math
import random
import time
from dataclasses import asdict, dataclass, replace

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

SEED = 7960
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def reset_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


plt.rcParams.update({
    "figure.figsize": (7, 4),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
    "axes.prop_cycle": plt.cycler(
        color=["#A31F34", "#1D4ED8", "#059669", "#B45309", "#7C3AED"]
    ),
})

reset_seed()
print(f"PyTorch {torch.__version__}; device = {DEVICE}")
if DEVICE.type == "cpu":
    print("A free Colab GPU is recommended for the ResNet-18 comparison.")


In [ ]:
QUICK_MODE = True
DATA_ROOT = "./data"

TARGET_TRAIN_PER_CLASS = 500
TARGET_VALIDATION_PER_CLASS = 100 if QUICK_MODE else 500
DEFAULT_BATCH_SIZE = 128
TRANSFER_EPOCHS = 2 if QUICK_MODE else 5
TRANSFER_IMAGE_SIZE = 96
NUM_WORKERS = 0 if QUICK_MODE else 2

print(
    f"quick_mode={QUICK_MODE}, target train={10 * TARGET_TRAIN_PER_CLASS:,}, "
    f"validation={10 * TARGET_VALIDATION_PER_CLASS:,}, "
    f"transfer epochs={TRANSFER_EPOCHS}"
)


## Setup: one fixed target split

Every run uses the same stratified indices and a deterministic validation view.
The source notebooks provide CIFAR loaders and training patterns; the ResNet-18
transfer workflow on a fixed CIFAR-10 subset is an explicit online-course
adaptation.


In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

transfer_train_transform = transforms.Compose(
    [
        transforms.RandomResizedCrop(TRANSFER_IMAGE_SIZE, scale=(0.75, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)
transfer_eval_transform = transforms.Compose(
    [
        transforms.Resize((TRANSFER_IMAGE_SIZE, TRANSFER_IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)

split_source = datasets.CIFAR10(DATA_ROOT, train=True, download=True)


def stratified_split(targets, train_per_class, validation_per_class, seed):
    target_tensor = torch.as_tensor(targets)
    generator = torch.Generator().manual_seed(seed)
    train_indices, validation_indices = [], []
    for class_id in range(10):
        class_indices = torch.where(target_tensor == class_id)[0]
        order = torch.randperm(len(class_indices), generator=generator)
        class_indices = class_indices[order]
        needed = train_per_class + validation_per_class
        train_indices.extend(class_indices[:train_per_class].tolist())
        validation_indices.extend(
            class_indices[train_per_class:needed].tolist()
        )
    return sorted(train_indices), sorted(validation_indices)


train_indices, validation_indices = stratified_split(
    split_source.targets,
    TARGET_TRAIN_PER_CLASS,
    TARGET_VALIDATION_PER_CLASS,
    SEED,
)
assert set(train_indices).isdisjoint(validation_indices)

transfer_train_source = datasets.CIFAR10(
    DATA_ROOT, train=True, transform=transfer_train_transform, download=False
)
transfer_eval_source = datasets.CIFAR10(
    DATA_ROOT, train=True, transform=transfer_eval_transform, download=False
)
transfer_train_set = Subset(transfer_train_source, train_indices)
transfer_train_eval_set = Subset(transfer_eval_source, train_indices)
transfer_validation_set = Subset(transfer_eval_source, validation_indices)


def seeded_loader(dataset, *, batch_size, shuffle, seed=SEED):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, drop_last=False,
        num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
        generator=generator,
    )


def trainable_parameter_count(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print("target-training examples:", len(train_indices))
print("validation examples:", len(validation_indices))
print("official CIFAR-10 test split: reserved and not loaded")


<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 1</div>
<div style="font-weight:700;font-size:1.2rem;">Write <code>sgd_step</code></div>
</div>


Complete `sgd_step` for a single momentum update:

$$v \leftarrow \mu\, v + g, \qquad \theta \leftarrow \theta - \eta\, v.$$

Return the updated parameter and velocity. The function must work for plain
Python floats and for tensors, so use ordinary arithmetic operators.


<details style="border:1px solid #e5e7eb;border-radius:8px;padding:10px 14px;background:#f9fafb;color:#111827;margin:14px 0;">
<summary style="cursor:pointer;font-weight:600;">Stuck on the return?</summary>
<div style="margin-top:8px;">Return the pair in the order the checks expect, <code>(parameter, velocity)</code>. Since the body uses only <code>+</code>, <code>*</code>, and <code>-</code>, the same function serves the float probe here and the tensors during training.</div>
</details>


In [ ]:
def sgd_step(parameter, gradient, velocity, learning_rate, momentum):
    """Return (updated parameter, updated velocity) for one momentum step."""
    # STUDENT TASK 1: implement momentum SGD.
    #   velocity  <- momentum * velocity + gradient
    #   parameter <- parameter - learning_rate * velocity
    raise NotImplementedError("Return the updated (parameter, velocity).")


In [ ]:
sgd_probe_parameter = 2.0
sgd_probe_velocity = 0.0
for _ in range(3):
    sgd_probe_parameter, sgd_probe_velocity = sgd_step(
        sgd_probe_parameter, 0.5, sgd_probe_velocity,
        learning_rate=0.1, momentum=0.9,
    )
sgd_probe_parameter = round(sgd_probe_parameter, 4)
sgd_probe_contract = (
    abs(sgd_probe_parameter - 1.7195) < 1e-6
    and abs(sgd_probe_velocity - 1.355) < 1e-6
)
assert sgd_probe_contract, "Check the momentum accumulation and update order."
print("Task 1 checks passed. Probe parameter after three steps:",
      sgd_probe_parameter)


<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 2</div>
<div style="font-weight:700;font-size:1.2rem;">Write <code>build_transfer_model</code></div>
</div>


Complete `build_transfer_model` so it (1) reads `backbone.fc.in_features`,
(2) freezes every existing backbone parameter, and (3) replaces `backbone.fc`
with a fresh `nn.Linear(in_features, num_classes)` whose parameters train. The
new head's parameters are trainable by default.


In [ ]:
def build_transfer_model(backbone, num_classes):
    """Freeze the backbone and attach a fresh trainable classification head."""
    # STUDENT TASK 2:
    #   in_features = backbone.fc.in_features
    #   freeze every current parameter, then set backbone.fc to a new Linear
    raise NotImplementedError(
        "Return the backbone with a fresh, trainable fc head."
    )


In [ ]:
class ProbeBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.body = nn.Linear(3, 512)
        self.fc = nn.Linear(512, 1000)

    def forward(self, x):
        return self.fc(torch.relu(self.body(x)))


probe_backbone = build_transfer_model(ProbeBackbone(), num_classes=10)
transfer_probe_count = trainable_parameter_count(probe_backbone)
transfer_probe_contract = (
    isinstance(probe_backbone.fc, nn.Linear)
    and probe_backbone.fc.in_features == 512
    and probe_backbone.fc.out_features == 10
    and not probe_backbone.body.weight.requires_grad
    and transfer_probe_count == 512 * 10 + 10
)
assert transfer_probe_contract, "Only the new 512->10 head should train."
assert transfer_probe_count == 5130
print("Task 2 checks passed. Trainable head parameters:", transfer_probe_count)


<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 3</div>
<div style="font-weight:700;font-size:1.2rem;">Write <code>batchnorm_forward</code></div>
</div>


Complete `batchnorm_forward`. Standardize each feature over the batch dimension
using the population variance (`unbiased=False`), then apply the affine scale
and shift: return `gamma * (x - mean) / sqrt(var + eps) + beta`.


In [ ]:
def batchnorm_forward(inputs, gamma, beta, eps):
    """Standardize per feature over the batch, then scale and shift."""
    # STUDENT TASK 3:
    #   mean/var over dim 0 (population variance, unbiased=False)
    #   normalized = (inputs - mean) / sqrt(var + eps)
    #   return gamma * normalized + beta
    raise NotImplementedError("Return the affine batch-normalized activations.")


In [ ]:
bn_probe_input = torch.tensor([[2.0], [4.0], [4.0], [6.0]])
bn_probe_output = batchnorm_forward(bn_probe_input, gamma=1.0, beta=0.0, eps=0.0)
batchnorm_probe_value = round(bn_probe_output[-1].item(), 5)
batchnorm_probe_contract = (
    abs(bn_probe_output.mean().item()) < 1e-6
    and abs(bn_probe_output.std(unbiased=False).item() - 1.0) < 1e-6
    and abs(batchnorm_probe_value - 1.41421) < 1e-4
)
assert batchnorm_probe_contract, "Output should have mean 0 and unit variance."
print("Task 3 checks passed. Standardized probe activation:", batchnorm_probe_value)


## Provided: training and transfer rails

These rails assemble a ResNet-18 from a configuration, train only the parameters
you leave unfrozen, and record complete histories. `assemble_model` calls your
`build_transfer_model` whenever the head is a single linear layer.


In [ ]:
@dataclass(frozen=True)
class TransferExperiment:
    use_pretrained: bool = True
    head_hidden: int = 0
    freeze_backbone: bool = True
    weight_decay: float = 0.0
    use_augmentation: bool = False
    learning_rate: float = 1e-3
    batch_size: int = DEFAULT_BATCH_SIZE
    epochs: int = TRANSFER_EPOCHS
    seed: int = SEED


def changed_config_fields(reference, candidate):
    reference_fields = asdict(reference)
    candidate_fields = asdict(candidate)
    return {
        name for name in reference_fields
        if reference_fields[name] != candidate_fields[name]
    }


def history_is_complete(record):
    history = record["history"]
    epochs = record["config"]["epochs"]
    return all(
        len(history[metric]) == epochs
        for metric in (
            "train_loss", "train_accuracy",
            "validation_loss", "validation_accuracy",
        )
    )


def build_head(in_features, head_hidden, num_classes):
    # The classifier head on top of the frozen 512-d features: one linear layer,
    # or a two-layer MLP once head_hidden > 0 (your task-4 design).
    if head_hidden <= 0:
        return nn.Linear(in_features, num_classes)
    return nn.Sequential(
        nn.Linear(in_features, head_hidden),
        nn.ReLU(),
        nn.Linear(head_hidden, num_classes),
    )


def assemble_model(config):
    # Build the ResNet-18 a config calls for: ImageNet-pretrained or random
    # weights, then attach the head and set which parameters train.
    reset_seed(config.seed)
    weights = (
        models.ResNet18_Weights.IMAGENET1K_V1
        if config.use_pretrained else None
    )
    backbone = models.resnet18(weights=weights)
    if config.head_hidden <= 0:
        # Linear head: your build_transfer_model freezes the backbone and swaps
        # in a fresh 512->10 layer; unfreeze again for full fine-tuning.
        backbone = build_transfer_model(backbone, num_classes=10)
        if not config.freeze_backbone:
            for parameter in backbone.parameters():
                parameter.requires_grad = True
    else:
        # MLP head: freeze (or not) the backbone, then attach your wider head.
        in_features = backbone.fc.in_features
        for parameter in backbone.parameters():
            parameter.requires_grad = not config.freeze_backbone
        backbone.fc = build_head(in_features, config.head_hidden, 10)
    return backbone


@torch.no_grad()
def evaluate_classifier(model, loader):
    # Mean loss and accuracy over a loader: sum across all examples, divide once.
    model.eval()
    loss_fn = nn.CrossEntropyLoss(reduction="sum")
    total_loss = total_correct = total_examples = 0
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        logits = model(images)
        total_loss += loss_fn(logits, labels).item()
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_examples += labels.numel()
    return total_loss / total_examples, total_correct / total_examples


def plot_history(record):
    history = record["history"]
    epochs = np.arange(1, len(history["train_accuracy"]) + 1)
    plt.figure(figsize=(5, 3))
    plt.plot(epochs, history["train_accuracy"], marker="o", label="train")
    plt.plot(epochs, history["validation_accuracy"], marker="o", label="val")
    plt.title(record["label"]); plt.xlabel("epoch"); plt.ylabel("accuracy")
    plt.ylim(0, 1); plt.legend(); plt.tight_layout(); plt.show()


def run_transfer_experiment(label, config):
    # Train one configuration end to end and return a record with its config (as
    # a dict) and the full per-epoch history that the report cell later checks.
    model = assemble_model(config).to(DEVICE)
    # Augmented view only when the config asks for it; evaluation view otherwise.
    train_source = (
        transfer_train_set if config.use_augmentation
        else transfer_train_eval_set
    )
    fit_loader = seeded_loader(
        train_source, batch_size=config.batch_size, shuffle=True,
        seed=config.seed,
    )
    train_eval_loader = seeded_loader(
        transfer_train_eval_set, batch_size=config.batch_size,
        shuffle=False, seed=config.seed,
    )
    validation_loader = seeded_loader(
        transfer_validation_set, batch_size=config.batch_size,
        shuffle=False, seed=config.seed,
    )
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(
        trainable, lr=config.learning_rate, weight_decay=config.weight_decay
    )
    loss_fn = nn.CrossEntropyLoss()
    history = {
        "train_loss": [], "train_accuracy": [],
        "validation_loss": [], "validation_accuracy": [],
    }
    for epoch in range(1, config.epochs + 1):
        model.train()
        for images, labels in fit_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(images), labels)
            loss.backward()
            optimizer.step()
        train_loss, train_accuracy = evaluate_classifier(model, train_eval_loader)
        validation_loss, validation_accuracy = evaluate_classifier(
            model, validation_loader
        )
        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["validation_loss"].append(validation_loss)
        history["validation_accuracy"].append(validation_accuracy)
        print(
            f"{label:>24} | epoch {epoch}/{config.epochs} | "
            f"val acc {validation_accuracy:.2%}"
        )
    record = {"label": label, "config": asdict(config), "history": history}
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return record


control_config = TransferExperiment()
control_result = run_transfer_experiment("transfer control", control_config)
plot_history(control_result)


<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 4</div>
<div style="font-weight:700;font-size:1.2rem;">Design a classifier head</div>
</div>


Choose `head_hidden` so that a two-layer head on top of the frozen 512-d
features has **20,000 to 120,000** trainable parameters. The head has
`513 * head_hidden + 10 * head_hidden + 10 = 523 * head_hidden + 10` parameters.


In [ ]:
# STUDENT TASK 4: pick a positive head_hidden inside the budget.
head_hidden = 0

learner_head = build_head(512, head_hidden, num_classes=10)
learner_head_parameter_count = trainable_parameter_count(learner_head)
learner_parameter_count = learner_head_parameter_count
if not (20_000 <= learner_parameter_count <= 120_000):
    raise ValueError(
        "Choose head_hidden so the head has 20,000-120,000 parameters."
    )
learner_config = replace(control_config, head_hidden=head_hidden)
print(f"learner head parameters: {learner_parameter_count:,}")


In [ ]:
scratch_config = replace(control_config, use_pretrained=False)
scratch_result = run_transfer_experiment("from-scratch anchor", scratch_config)
learner_result = run_transfer_experiment("learner head design", learner_config)
for record in (control_result, scratch_result, learner_result):
    plot_history(record)


<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 5</div>
<div style="font-weight:700;font-size:1.2rem;">Choose one fine-tuning intervention</div>
</div>


Predict what will change, then choose exactly one intervention relative to the
transfer control: unfreeze the backbone (`full_finetune`), add training
augmentation (`augmentation`), or add weight decay (`weight_decay`). The
contract permits exactly one changed field.


In [ ]:
# STUDENT TASK 5: choose "full_finetune", "augmentation", or "weight_decay".
FINE_TUNE_CHOICE = ""

TREATMENT_OVERRIDES = {
    "full_finetune": {"freeze_backbone": False},
    "augmentation": {"use_augmentation": True},
    "weight_decay": {"weight_decay": 1e-4},
}
TREATMENT_FIELDS = {
    "full_finetune": "freeze_backbone",
    "augmentation": "use_augmentation",
    "weight_decay": "weight_decay",
}
TREATMENT_LABELS = {
    "full_finetune": "chosen treatment: full fine-tune",
    "augmentation": "chosen treatment: augmentation",
    "weight_decay": "chosen treatment: weight decay",
}
if FINE_TUNE_CHOICE not in TREATMENT_OVERRIDES:
    raise ValueError("Choose full_finetune, augmentation, or weight_decay.")
treatment_config = replace(
    control_config, **TREATMENT_OVERRIDES[FINE_TUNE_CHOICE]
)
assert changed_config_fields(control_config, treatment_config) == {
    TREATMENT_FIELDS[FINE_TUNE_CHOICE]
}


In [ ]:
treatment_result = run_transfer_experiment(
    TREATMENT_LABELS[FINE_TUNE_CHOICE], treatment_config
)
for record in (control_result, treatment_result):
    plot_history(record)


## Pause and reflect (ungraded)

Which comparison most confirmed or changed your prediction? Pick one pattern in
the curves worth explaining, name a reason it might mislead you, and sketch the
one-factor run you would try next. Nothing here is submitted.


## Report values

Run the cell below after finishing all five tasks and all four training runs.
R1 through R3 confirm your three functions on fixed probes. R4 is the parameter
count of your own valid head design, so different learners report different
values inside the 20,000 to 120,000 range. R5 confirms the disjoint split, the
one-factor comparisons, and the four completed histories.

Enter R1 through R5 in the first Lab 2 submission problem.

**Before you submit**

- [ ] I entered R1 through R5 from the report cell.
- [ ] I answered the transfer-advantage question.
- [ ] I answered the fine-tuning-intervention question.
- [ ] I did not upload code or submit a measured accuracy as an exact value.


In [ ]:
run_config_pairs = (
    (control_result, control_config),
    (scratch_result, scratch_config),
    (learner_result, learner_config),
    (treatment_result, treatment_config),
)

workflow_contract = int(
    set(train_indices).isdisjoint(validation_indices)
    and len({record["label"] for record, _ in run_config_pairs}) == 4
    and all(
        record["config"] == asdict(config)
        for record, config in run_config_pairs
    )
    and all(history_is_complete(record) for record, _ in run_config_pairs)
    and changed_config_fields(control_config, scratch_config)
        == {"use_pretrained"}
    and changed_config_fields(control_config, learner_config)
        == {"head_hidden"}
    and changed_config_fields(control_config, treatment_config)
        == {TREATMENT_FIELDS[FINE_TUNE_CHOICE]}
    and learner_parameter_count == learner_head_parameter_count
    and 20_000 <= learner_parameter_count <= 120_000
)

probe_sgd_parameter = round(sgd_probe_parameter, 4) if sgd_probe_contract else -1.0
probe_transfer_parameters = (
    transfer_probe_count if transfer_probe_contract else -1
)
probe_batchnorm_value = (
    round(batchnorm_probe_value, 5) if batchnorm_probe_contract else -1.0
)

report_values = {
    "R1: momentum-SGD probe parameter after three steps": probe_sgd_parameter,
    "R2: transfer head trainable parameters (frozen backbone)": probe_transfer_parameters,
    "R3: batch-normalized probe activation": probe_batchnorm_value,
    "R4: learner-designed head parameter count": learner_parameter_count,
    "R5: experimental workflow contract": workflow_contract,
}
assert probe_sgd_parameter == 1.7195
assert probe_transfer_parameters == 5130
assert probe_batchnorm_value == 1.41421
assert 20_000 <= learner_parameter_count <= 120_000
assert workflow_contract == 1

print("LAB 2 REPORT VALUES")
for label, value in report_values.items():
    print(f"{label}: {value}")
